# Initialize: grid pursuit-evasion benchmark
Defines the environment, the latent context, the defender's two-part representation, the stage-game solver, the networks, the training updates, the baseline architectures, and the defender wrappers used by the other notebooks.

Conventions:
- **Game.** A pursuer (the defender, Player 1) chases an evader (the attacker, Player 2) on a 6 × 6 grid for at most $T = 40$ rounds. Both move simultaneously (up, right, down, left). A move into the grid edge or into an obstacle leaves the player in place. Capture (same cell, or the two swapping cells) ends the episode.
- **Payoff.** `Reward` returns the **pursuer's** payoff: +4 for a capture off the exit, +2 for a capture on the exit, −4 if the evader stands on its exit uncaptured at the horizon, and otherwise a distance shaping term $0.05\,(10 - \lVert e - p\rVert_1)$, minus 0.25 while the evader is on its exit. The critic learns the pursuer's payoff, and the pursuer is the row maximizer in the stage game; this is the paper's convention with the sign flipped.
- **Latent context** $\theta = (\theta^{\mathrm{env}}, \theta^{\mathrm{type}})$, drawn once per episode by `sample_config`:
  - the obstacle layout, one of 20 four-cell patterns in `LAYOUT_BANK`, unknown to both players, and
  - the evader's exit, one of the four corners, known only to the evader.
- **Observations.** The pursuer sees its own position and a signal that places the evader in a random 2 × 2 block containing it.
- **Representation** $z_k = [z^S_k, z^\Theta_k]$:
  - the state component $z^S$ is the pursuer's own position plus a Bayes filter over the evader's cell (`Def_State_Blf`), with uniform evader moves and no knowledge of the layout;
  - the context component $z^\Theta$ is a Transformer over the history rows $[z^S_{k-1}, a^1_{k-1}, z^S_k]$ (`TSTransformer`).
- **Evader's private state** $m^2_k$: its belief over which cells are obstacles, updated from its own blocked moves (`m2_update`).
- **Encoder objective**, Eq. (3): $\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$ (`Train_Encoder`).
- **Game step.** The critic is trained by TD regression onto the minimax value of the next stage game, with terminal transitions masked, and both actors are distilled toward its equilibrium strategies by cross-entropy. Embeddings are recomputed with the current encoder at every update.

In [ ]:
import os
import copy
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import linprog
from collections import deque
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.distributions import Categorical

torch.backends.mha.set_fastpath_enabled(False)      # keeps the padding-mask path identical in train and eval
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Benchmark definition: 6 × 6 grid, 20 layouts, 4 exits, 40 rounds

In [ ]:
#      0   1   2   3   4   5
#     ------------------------
#  0 | 1   2   3   4   5   6
#  1 | 7   8   9   10  11  12
#  2 | 13  14  15  16  17  18
#  3 | 19  20  21  22  23  24
#  4 | 25  26  27  28  29  30
#  5 | 31  32  33  34  35  36
LAYOUT_BANK = [
    [(1,1),(2,1),(3,1),(4,1)],        # 0  vertical wall, col1
    [(1,4),(2,4),(3,4),(4,4)],        # 1  vertical wall, col4
    [(1,1),(1,2),(1,3),(1,4)],        # 2  horizontal wall, row1
    [(4,1),(4,2),(4,3),(4,4)],        # 3  horizontal wall, row4
    [(2,1),(2,2),(2,3),(2,4)],        # 4  horizontal wall, row2 (mid)
    [(3,1),(3,2),(3,3),(3,4)],        # 5  horizontal wall, row3 (mid)
    [(1,2),(2,2),(3,2),(4,2)],        # 6  vertical wall, col2
    [(1,3),(2,3),(3,3),(4,3)],        # 7  vertical wall, col3
    [(2,2),(2,3),(3,2),(3,3)],        # 8  central 2x2 block
    [(1,1),(2,2),(3,3),(4,4)],        # 9  main diagonal NW-SE
    [(1,4),(2,3),(3,2),(4,1)],        # 10 anti-diagonal NE-SW
    [(0,2),(1,2),(2,2),(2,1)],        # 11 NW chokepoint (guards exit (0,0))
    [(0,3),(1,3),(2,3),(2,4)],        # 12 NE chokepoint (guards exit (0,5))
    [(5,2),(4,2),(3,2),(3,1)],        # 13 SW chokepoint (guards exit (5,0))
    [(5,3),(4,3),(3,3),(3,4)],        # 14 SE chokepoint (guards exit (5,5))
    [(1,1),(1,2),(3,3),(3,4)],        # 15 staggered double barrier
    [(2,1),(2,2),(2,4),(3,3)],        # 16 broken mid wall
    [(1,2),(2,2),(4,3),(3,3)],        # 17 offset vertical pair
    [(2,2),(2,3),(3,3),(4,3)],        # 18 L-shaped barrier
    [(1,3),(2,2),(3,3),(4,2)],]       # 19 zigzag
EXITS = [(0, 0), (0, 5), (5, 0), (5, 5)]

d = 36                    # cells
n_a = 4                   # 0: up, 1: right, 2: down, 3: left
n_u = 4
Len_Theta = 40            # theta = [obstacle cells (36), exit one-hot (4)]
Embedding_Size = 64
max_steps = 40            # rounds per episode, also the context window
CTX_DIM = d * 4 + n_a     # history row: [z_s (72), action one-hot (4), next z_s (72)]


def sample_config():
    # Episode context (layout and exit) and initial positions, in the same order as main.ipynb
    layout_idx = np.random.randint(0, len(LAYOUT_BANK))
    Epi_Layout = set(LAYOUT_BANK[layout_idx])
    exit_corner = np.random.randint(0, 4)
    exit_rc = EXITS[exit_corner]
    p_row = np.random.randint(0, 6); p_col = np.random.randint(0, 6)
    while (p_row, p_col) in Epi_Layout:
        p_row = np.random.randint(0, 6); p_col = np.random.randint(0, 6)
    e_row = np.random.randint(0, 6); e_col = np.random.randint(0, 6)
    while (e_row, e_col) in Epi_Layout or (e_row, e_col) == (p_row, p_col) or (e_row, e_col) == exit_rc:
        e_row = np.random.randint(0, 6); e_col = np.random.randint(0, 6)
    Theta, Type_Theta = theta_of(Epi_Layout, exit_corner)
    return Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col


def theta_of(Epi_Layout, exit_corner):
    Theta = np.zeros(Len_Theta)
    for (r, c) in Epi_Layout:
        Theta[r * 6 + c] = 1.0
    Type_Theta = np.zeros(4)
    Type_Theta[exit_corner] = 1.0
    Theta[36:40] = Type_Theta
    return Theta, Type_Theta


def build_x(p_row, p_col, e_row, e_col):
    x = np.zeros(d * 2)
    x[p_row * 6 + p_col] = 1.0
    x[d + e_row * 6 + e_col] = 1.0
    return x


def init_z_s(p_row, p_col):
    # Pursuer's own position and a uniform belief over the evader's cell
    z_s = np.zeros(d * 2)
    z_s[p_row * 6 + p_col] = 1.0
    z_s[d:d * 2] = np.ones(d) / d
    return z_s


def init_m2(e_row, e_col):
    # Evader's prior over obstacles: 4 obstacles among the 31 cells that are neither corners nor its start
    m2 = np.full(d, 4 / 31)
    m2[e_row * 6 + e_col] = 0
    m2[0] = 0; m2[5] = 0; m2[30] = 0; m2[35] = 0
    return m2


def step_pos(r, c, action):
    nr, nc = r, c
    if   action == 0 and r > 0: nr = r - 1
    elif action == 1 and c < 5: nc = c + 1
    elif action == 2 and r < 5: nr = r + 1
    elif action == 3 and c > 0: nc = c - 1
    return nr, nc

## Dynamics, payoff, observations, pursuer's state filter, evader's private state

In [ ]:
def next_state(p_row, p_col, e_row, e_col, a, u, Layout):
    out = []
    for r, c, action in ((p_row, p_col, a), (e_row, e_col, u)):
        nr, nc = step_pos(r, c, action)
        if (nr, nc) in Layout:
            nr, nc = r, c
        out.extend((nr, nc))
    return tuple(out)
#################################################################################################################
def is_capture(p, e, new_p, new_e):
    # Same cell after the move, or the two players swapping cells
    return new_p == new_e or (p == new_e and e == new_p)
#################################################################################################################
def Reward(p_row, p_col, e_row, e_col, exit_rc, Max_Length_Reached, Captured):
    # Pursuer's payoff
    Envader_At_Exit = (e_row, e_col) == exit_rc
    if Envader_At_Exit and Max_Length_Reached and (not Captured):   # evader escaped
        return -4
    if Envader_At_Exit and Captured:                                # capture on the exit cell
        return +2
    if (not Envader_At_Exit) and Captured:                          # capture anywhere else
        return +4
    r = 0.05 * (10 - abs(e_row - p_row) - abs(e_col - p_col))      # not captured
    if Envader_At_Exit:
        r -= 0.25
    return r
#################################################################################################################
def Observation(new_e_row, new_e_col):
    r, c = new_e_row, new_e_col

    top_rows  = [br for br in (r-1, r) if 0 <= br <= 4]
    left_cols = [bc for bc in (c-1, c) if 0 <= bc <= 4]

    blocks = [(br, bc) for br in top_rows for bc in left_cols]
    br, bc = blocks[np.random.randint(len(blocks))]

    y = np.zeros(36)
    for dr in (0, 1):
        for dc in (0, 1):
            y[(br + dr) * 6 + (bc + dc)] = 0.25
    return y
#################################################################################################################
def Def_State_Blf(z_s, y, new_pers_r, new_pers_c):
    b = z_s[36:72]
    b_pred = np.zeros(36)
    for cell in range(36):
        if b[cell] == 0.0:
            continue
        r, c = cell // 6, cell % 6
        for action in range(4):
            nr, nc = r, c
            if   action == 0 and r > 0: nr = r - 1     # up
            elif action == 1 and c < 5: nc = c + 1     # right
            elif action == 2 and r < 5: nr = r + 1     # down
            elif action == 3 and c > 0: nc = c - 1     # left
            b_pred[nr*6 + nc] += b[cell] / 4.0

    b_meas = np.zeros(36)
    for e in np.where(y > 0)[0]:
        r, c = e // 6, e % 6
        nb = (2 if 0 < r < 5 else 1) * (2 if 0 < c < 5 else 1)
        b_meas[e] = b_pred[e] / nb

    s = b_meas.sum()
    if s > 0:
        b_new = b_meas / s
    elif b_pred.sum() > 0:
        b_new = b_pred / b_pred.sum()
    else:
        b_new = b

    next_z_s = np.zeros(72)
    next_z_s[new_pers_r * 6 + new_pers_c] = 1.0  
    next_z_s[36:72] = b_new
    return next_z_s
#################################################################################################################
def m2_update(m2, envd_row, envd_col, u, next_envd_row, next_envd_col):
    m2 = m2.copy()
    cur = envd_row * 6 + envd_col
    nxt = next_envd_row * 6 + next_envd_col
    ir, ic = envd_row, envd_col
    
    on_grid = True
    if   u == 0:  # up
        if envd_row > 0: ir = envd_row - 1
        else: on_grid = False
    elif u == 1:  # right
        if envd_col < 5: ic = envd_col + 1
        else: on_grid = False
    elif u == 2:  # down
        if envd_row < 5: ir = envd_row + 1
        else: on_grid = False
    elif u == 3:  # left
        if envd_col > 0: ic = envd_col - 1
        else: on_grid = False
    intended = ir * 6 + ic

    m2[cur] = 0.0
    if on_grid and (nxt != intended):
        m2[intended] = 1.0
    m2[nxt] = 0.0

    known_obs = int(np.sum(m2 == 1.0))
    unknown = (m2 != 1.0) & (m2 != 0.0)
    n_unknown = int(unknown.sum())
    if n_unknown > 0:
        m2[unknown] = (4 - known_obs) / n_unknown
    return m2
#################################################################################################################
def Context_Row(z_s, a, next_z_s):
    # One history row for the context encoder: [z^S_{k-1}, action one-hot, z^S_k]
    row = np.zeros(CTX_DIM, dtype=np.float32)
    row[0:2*d] = z_s
    row[2*d + a] = 1
    row[2*d + n_a:4*d + n_a] = next_z_s
    return row

def Context_Mask(num_rows):
    mask = torch.zeros((1, max_steps), dtype=torch.bool, device=device)
    mask[0, :max(1, min(num_rows, max_steps))] = True
    return mask

## Stage-game solver

In [ ]:
# Exact minimax by linear programming. Row player maximizes, column player minimizes.
# Returns (value, row strategy p, column strategy q).
def MatrixValue(M):
    M = np.asarray(M, dtype=float)
    if M.ndim != 2:
        raise ValueError("M must be a 2D array.")
    m, n = M.shape
    if m == 0 or n == 0 or not np.isfinite(M).all():
        return float(np.nanmean(M)), np.ones(m)/m if m else np.array([]), np.ones(n)/n if n else np.array([])
    # normalization
    center = float(M.mean())
    A = M - center
    scale = float(np.max(np.abs(A)))
    if scale == 0.0:  
        return center, np.ones(m)/m, np.ones(n)/n
    A /= scale  
    c1 = np.zeros(m + 1); c1[0] = -1.0  
    A_ub1 = np.zeros((n + m, m + 1))
    b_ub1 = np.zeros(n + m)
    A_ub1[:n, 0] =  1.0
    A_ub1[:n, 1:] = -A.T
    A_ub1[n:, 1:] = -np.eye(m)
    A_eq1 = np.ones((1, m + 1)); A_eq1[0, 0] = 0.0
    b_eq1 = np.array([1.0])
    bounds1 = [(None, None)] + [(0.0, None)] * m
    opt1 = linprog(c=c1, A_ub=A_ub1, b_ub=b_ub1, A_eq=A_eq1, b_eq=b_eq1,
                   bounds=bounds1, method="highs", options={"presolve": True})
    v_row = None; p = None
    if opt1.success and opt1.x is not None:
        v_row = float(opt1.x[0])
        p = opt1.x[1:]
    c2 = np.zeros(n + 1); c2[0] = 1.0 
    A_ub2 = np.zeros((m + n, n + 1))
    b_ub2 = np.zeros(m + n)
    A_ub2[:m, 0]  = -1.0
    A_ub2[:m, 1:] =  A
    A_ub2[m:, 1:] = -np.eye(n)
    A_eq2 = np.ones((1, n + 1)); A_eq2[0, 0] = 0.0
    b_eq2 = np.array([1.0])
    bounds2 = [(None, None)] + [(0.0, None)] * n
    opt2 = linprog(c=c2, A_ub=A_ub2, b_ub=b_ub2, A_eq=A_eq2, b_eq=b_eq2,
                   bounds=bounds2, method="highs", options={"presolve": True})
    v_col = None; q = None
    if opt2.success and opt2.x is not None:
        v_col = float(opt2.x[0])
        q = opt2.x[1:]
    if p is None and q is None:
        return center, np.ones(m)/m, np.ones(n)/n
    if p is None:
        p = np.ones(m)/m
        v_norm = v_col
    elif q is None:
        q = np.ones(n)/n
        v_norm = v_row
    else:
        v_norm = 0.5 * (v_row + v_col)
    p = np.clip(p, 0.0, 1.0); ps = p.sum(); p = p/ps if ps > 0 else np.ones(m)/m
    q = np.clip(q, 0.0, 1.0); qs = q.sum(); q = q/qs if qs > 0 else np.ones(n)/n
    value = center + scale * float(v_norm)
    return value, p, q

## Replay buffer and networks

In [ ]:
class ReplayMemory(object):
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, args):
        self.memory.append(args)

    def sample(self, batch_size):
        return random.sample(self.memory, batch_size)

    def __len__(self):
        return len(self.memory)
#################################################################################################################
# Context component z^Theta: Transformer with learned positional embeddings, masked mean pooling,
# feedforward head, and an l2-normalized output
class TSTransformer(nn.Module):
    def __init__(self):
        super(TSTransformer, self).__init__()
        self.input_projection = nn.Linear(d*4+n_a, 128)
        self.position_embeddings = nn.Embedding(max_steps, 128)
        encoder_layers = TransformerEncoderLayer(d_model=128, nhead=4, batch_first=True, dropout=0.1)
        self.transformer_encoder = TransformerEncoder(encoder_layers, num_layers=3, enable_nested_tensor=False)
        self.fc1 = nn.Linear(128, 128)
        self.norm1 = nn.LayerNorm(128)
        self.fc2 = nn.Linear(128, Embedding_Size)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)  

    def forward(self, x, mask):
        x = self.input_projection(x)  
        batch_size, seq_length, _ = x.shape 
        position_ids = torch.arange(seq_length, device=x.device).unsqueeze(0).expand(batch_size, -1)  
        position_embeddings = self.position_embeddings(position_ids)  
        x = x + position_embeddings
        key_padding_mask = ~mask.bool()
        x = self.transformer_encoder(x, src_key_padding_mask=key_padding_mask)
        mask = mask.unsqueeze(-1) 
        x = (x * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)  
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)  
        x = F.normalize(x, p=2, dim=1) 
        return x
#################################################################################################################
# Centralized Q(z^S, z^Theta, state, m^2, theta_type, theta) over all joint actions, the pursuer's payoff
class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.enc_zs1    = nn.Linear(d*2, 128)
        self.enc_zs2    = nn.Linear(128, 128)
        self.enc_embd1  = nn.Linear(Embedding_Size, 128)
        self.enc_embd2  = nn.Linear(128, 128)
        self.enc_state1 = nn.Linear(d*2, 128)
        self.enc_state2 = nn.Linear(128, 128)
        self.enc_m2_1   = nn.Linear(d, 128)
        self.enc_m2_2   = nn.Linear(128, 128)
        self.enc_type1  = nn.Linear(4, 64)
        self.enc_type2  = nn.Linear(64, 128)
        self.enc_theta1 = nn.Linear(Len_Theta, 64)
        self.enc_theta2 = nn.Linear(64, 128)

        self.ln_fuse = nn.LayerNorm(128 * 6)
        self.fuse1   = nn.Linear(128 * 6, 512)
        self.fuse2   = nn.Linear(512, 128)
        self.out     = nn.Linear(128, n_a * n_u)

    def forward(self, z_s, Embeding, x, m2, type_theta, theta):
        z_b = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        z_c = F.relu(self.enc_embd2(F.relu(self.enc_embd1(Embeding))))
        z_x = F.relu(self.enc_state2(F.relu(self.enc_state1(x))))
        z_m = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        z_y = F.relu(self.enc_type2(F.relu(self.enc_type1(type_theta))))
        z_t = F.relu(self.enc_theta2(F.relu(self.enc_theta1(theta))))
        e = torch.cat([z_b, z_c, z_x, z_m, z_y, z_t], dim=-1)
        e = self.ln_fuse(e)
        e = F.relu(self.fuse1(e))
        e = F.relu(self.fuse2(e))
        q = self.out(e)
        return q
#################################################################################################################
# beta(u | state, m^2, exit): the evader
class AttackerActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_x1   = nn.Linear(d*2, 128)
        self.enc_x2   = nn.Linear(128, 128)
        self.enc_m2_1 = nn.Linear(d, 128)
        self.enc_m2_2 = nn.Linear(128, 128)
        self.enc_ty1  = nn.Linear(4, 64)
        self.enc_ty2  = nn.Linear(64, 128)
        self.fuse1 = nn.Linear(384, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_u)

    def forward(self, x, m2, type_theta):
        zx = F.relu(self.enc_x2(F.relu(self.enc_x1(x))))
        zm = F.relu(self.enc_m2_2(F.relu(self.enc_m2_1(m2))))
        zt = F.relu(self.enc_ty2(F.relu(self.enc_ty1(type_theta))))
        h = torch.cat([zx, zm, zt], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        logits = self.head(h)
        return F.softmax(logits, dim=-1)  
#################################################################################################################
# pi^1(a | z), with z = [z^S, z^Theta]
class DefenderActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_zs1  = nn.Linear(d*2, 128)
        self.enc_zs2  = nn.Linear(128, 128)
        self.enc_emb1 = nn.Linear(Embedding_Size, 128)
        self.enc_emb2 = nn.Linear(128, 128)
        self.fuse1 = nn.Linear(256, 256)
        self.ln    = nn.LayerNorm(256)
        self.fuse2 = nn.Linear(256, 128)
        self.head  = nn.Linear(128, n_a)

    def forward(self, z_s, Embeding):
        zb = F.relu(self.enc_zs2(F.relu(self.enc_zs1(z_s))))
        zc = F.relu(self.enc_emb2(F.relu(self.enc_emb1(Embeding))))
        h = torch.cat([zb, zc], dim=-1)
        h = self.ln(F.relu(self.fuse1(h)))
        h = F.relu(self.fuse2(h))
        logits = self.head(h)
        return F.softmax(logits, dim=-1)

## Representation step: encoder objective, Eq. (3)
$\mathcal L_{\mathrm{enc}} = \lambda_r \mathcal L_r + \lambda_p \mathcal L_p + \lambda_\Theta \mathcal L_{\mathrm{SI}}$:
- $\mathcal L_r$: regression of the stage reward on the representation, through a reward head.
- $\mathcal L_p$: prediction of the next representation from $(z_k, a^1_k, y_{k+1})$ through a latent transition model and a prediction head, against gradient-free exponential-moving-average copies of the encoder and projection. `F.mse_loss` averages over the 128 projection dimensions, so it is the squared norm in Eq. (3) divided by 128; the constant is absorbed into $\lambda_p$.
- $\mathcal L_{\mathrm{SI}}$: Soft-InfoNCE on the $\ell_2$-normalized projection, $\mathcal L_i = -\sum_{j\ne i} w_{ij}\log p_{ij}$, with $p_{ij}$ a softmax of $g_i^\top g_j/T$ over $k \ne i$ and graded targets $w_{ij} = s_{ij}/\sum_{\ell\ne i}s_{i\ell}$, $s_{ij} = \exp(-\tilde d_\Theta(\theta_i,\theta_j)/\sigma_\Theta)$.

The computable proxy is $\tilde d_\Theta = d_{\mathrm{exit}} + d_{\mathrm{layout}}$:
- $d_{\mathrm{exit}} = 1$ if the exits differ and 0 otherwise, and
- $d_{\mathrm{layout}} = 1 - J$, with $J$ the Jaccard similarity of the two obstacle sets (`Context_Similarity`).

In [ ]:
def Context_Similarity(th):
    # s_ij = exp(-(d_exit + d_layout) / sigma_Theta)
    obs = th[:, :36]
    ext = th[:, 36:40]
    inter = obs @ obs.t()
    layout_sim = inter / (8.0 - inter).clamp(min=1e-9)          # each layout has 4 cells
    exit_sim = ext @ ext.t()
    return torch.exp(-((1.0 - exit_sim) + (1.0 - layout_sim)) / SIGMA_THETA)
#################################################################################################################
def ema_update(tau=0.995):
    # Gradient-free targets for the next-representation loss
    for po, pt in zip(Context_Model.parameters(), target_model.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for po, pt in zip(proj_online.parameters(), proj_target.parameters()):
        pt.data.lerp_(po.data, 1.0 - tau)
    for bo, bt in zip(Context_Model.buffers(), target_model.buffers()):
        bt.data.copy_(bo.data)
    for bo, bt in zip(proj_online.buffers(), proj_target.buffers()):
        bt.data.copy_(bo.data)
#################################################################################################################
def Train_Encoder():
    if len(context_memory) < CONTEXT_BATCH:
        return 0.0
    batch = context_memory.sample(CONTEXT_BATCH)
    (ctx, msk, nctx, nmsk, a, y, r, th) = map(list, zip(*batch))
    ctx  = torch.cat(ctx,  dim=0).to(device)
    msk  = torch.cat(msk,  dim=0).to(device)
    nctx = torch.cat(nctx, dim=0).to(device)
    nmsk = torch.cat(nmsk, dim=0).to(device)
    a  = torch.stack(a).to(device)
    y  = torch.stack(y).to(device).float()
    r  = torch.stack(r).to(device).float()
    th = torch.stack(th).to(device).float()
    Bsz = ctx.shape[0]

    Context_Model.train(); trans_pred.train(); reward_head.train()
    proj_online.train();  pred_online.train()
    z = Context_Model(ctx, msk)

    # L_r: reward prediction
    loss_r = F.mse_loss(reward_head(z).squeeze(-1), r)

    # L_p: next-representation prediction against the EMA target encoder and projection
    z_hat = trans_pred(torch.cat([z, F.one_hot(a, num_classes=n_a).float(), y], dim=-1))
    h_hat = F.normalize(pred_online(proj_online(z_hat)), p=2, dim=1)
    with torch.no_grad():
        h_tgt = F.normalize(proj_target(target_model(nctx, nmsk)), p=2, dim=1)
    loss_p = F.mse_loss(h_hat, h_tgt)

    # L_SI: Soft-InfoNCE with graded targets from the context similarity
    g = F.normalize(proj_online(z), p=2, dim=1)
    eye = torch.eye(Bsz, device=device, dtype=torch.bool)
    logits = ((g @ g.t()) / SI_TEMPERATURE).masked_fill(eye, -1e9)
    s = Context_Similarity(th).masked_fill(eye, 0.0)
    w = s / s.sum(dim=1, keepdim=True).clamp(min=1e-8)
    loss_si = -(w * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

    loss = LAMBDA_R * loss_r + LAMBDA_P * loss_p + LAMBDA_THETA * loss_si

    encoder_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        list(Context_Model.parameters()) + list(trans_pred.parameters()) +
        list(reward_head.parameters())  + list(proj_online.parameters()) +
        list(pred_online.parameters()), max_norm=1.0)
    encoder_optimizer.step()
    ema_update()

    Context_Model.eval()
    return loss.item()

## Game step: critic regression and actor distillation

In [ ]:
def _game_batch(batch_size):
    # Sample transitions and recompute the context embeddings with the current encoder
    batch = exp_memory.sample(batch_size)
    (x, nx, a, u, r, done, tt, th, m2, nm2, zs, nzs, ctx, msk, nctx, nmsk) = map(list, zip(*batch))
    S = lambda v: torch.stack(v).to(device)
    C = lambda v: torch.cat(v, dim=0).to(device)
    Context_Model.eval()
    with torch.no_grad():
        emb  = Context_Model(C(ctx), C(msk))
        nemb = Context_Model(C(nctx), C(nmsk))
    return dict(x=S(x), nx=S(nx), a=S(a), u=S(u), r=S(r), done=S(done), tt=S(tt), th=S(th),
                m2=S(m2), nm2=S(nm2), zs=S(zs), nzs=S(nzs), emb=emb, nemb=nemb)
#################################################################################################################
def Train_Critic():
    if len(exp_memory) < CRITIC_BATCH:
        return 0.0
    b = _game_batch(CRITIC_BATCH)
    Bsz = b['x'].shape[0]

    q_flat   = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th'])
    q_values = q_flat.gather(1, (b['a'] * n_u + b['u']).unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = critic_target(b['nzs'], b['nemb'], b['nx'], b['nm2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        V_next = torch.tensor([MatrixValue(next_q[i])[0] for i in range(Bsz)], dtype=torch.float32, device=device)
        target_values = b['r'] + GAMMA * (1 - b['done'].float()) * V_next   # terminal transitions masked

    loss = F.smooth_l1_loss(q_values, target_values)
    critic_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=1.0)
    critic_optimizer.step()

    with torch.no_grad():
        for tgt, src in zip(critic_target.parameters(), critic.parameters()):
            tgt.data.mul_(1.0 - TAU).add_(TAU * src.data)
    return loss.item()
#################################################################################################################
def Train_Player1():
    # Pursuer: cross-entropy toward the row (maximizing) equilibrium strategy of the critic's stage game
    if len(exp_memory) < PLAYER1_BATCH:
        return 0.0
    b = _game_batch(PLAYER1_BATCH)
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_a = torch.tensor(np.stack([MatrixValue(Q[i])[1] for i in range(Bsz)]), dtype=torch.float32, device=device)
    pred_a = Player1(b['zs'], b['emb'])
    loss = -(tgt_a * torch.log(pred_a + 1e-8)).sum(dim=1).mean()
    player1_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player1.parameters(), max_norm=1.0)
    player1_optimizer.step()
    return loss.item()
#################################################################################################################
def Train_Player2():
    # Evader: cross-entropy toward the column (minimizing) equilibrium strategy, acting on its private information
    if len(exp_memory) < PLAYER2_BATCH:
        return 0.0
    b = _game_batch(PLAYER2_BATCH)
    Bsz = b['x'].shape[0]
    with torch.no_grad():
        Q = critic(b['zs'], b['emb'], b['x'], b['m2'], b['tt'], b['th']).view(Bsz, n_a, n_u).cpu().numpy()
        tgt_u = torch.tensor(np.stack([MatrixValue(Q[i])[2] for i in range(Bsz)]), dtype=torch.float32, device=device)
    pred_u = Player2(b['x'], b['m2'], b['tt'])
    loss = -(tgt_u * torch.log(pred_u + 1e-8)).sum(dim=1).mean()
    player2_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(Player2.parameters(), max_norm=1.0)
    player2_optimizer.step()
    return loss.item()

## Baseline architectures and the best-response attacker (shared by baselines, best_response, eval, and outcomes)

In [ ]:
OBS_DIM = 2 * d + n_a   # baseline observation: [own position, y, previous action]
RNN_H = 128
LATENT_DIM = 16


class RecurrentAC(nn.Module):
    # Recurrent actor-critic over the baseline observation, optionally conditioned on an extra input
    def __init__(self, extra_dim=0, rnn_h=RNN_H):
        super().__init__()
        self.extra_dim = extra_dim
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        dd = rnn_h + extra_dim
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_a))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, extra=None, h0=None):
        out, hT = self.gru(obs, h0)
        feat = out if extra is None else torch.cat([out, extra], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT
#################################################################################################################
class VariBADEncoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(OBS_DIM, rnn_h, batch_first=True)
        self.mu = nn.Linear(rnn_h, latent_dim)
        self.logvar = nn.Linear(rnn_h, latent_dim)
    def forward(self, obs, h0=None):
        out, hT = self.gru(obs, h0)
        return self.mu(out), self.logvar(out), hT
#################################################################################################################
class PEARLEncoder(nn.Module):
    # Product-of-Gaussians posterior over the latent from the episode's transitions (y, a, y_next)
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(nn.Linear(2 * d + n_a, 128), nn.ReLU(), nn.Linear(128, 2 * latent_dim))
    def factors(self, trans):
        out = self.net(trans)
        mu = out[..., :self.latent_dim]
        var = F.softplus(out[..., self.latent_dim:]) + 1e-6
        return mu, var
    def posterior(self, trans):
        if trans.shape[0] == 0:
            return torch.zeros(self.latent_dim, device=device), torch.ones(self.latent_dim, device=device)
        mu_i, var_i = self.factors(trans)
        prec = 1.0 / var_i
        prec_post = 1.0 + prec.sum(0)
        mu_post = (prec * mu_i).sum(0) / prec_post
        return mu_post, 1.0 / prec_post
#################################################################################################################
class BRAttacker(nn.Module):
    # Recurrent best-response evader: GRU over [true state, previous move one-hot], conditioned on the full theta
    def __init__(self, rnn_h=RNN_H):
        super().__init__()
        self.gru = nn.GRU(2 * d + n_u, rnn_h, batch_first=True)
        self.theta_enc = nn.Sequential(nn.Linear(Len_Theta, 64), nn.ReLU())
        dd = rnn_h + 64
        self.pi = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, n_u))
        self.v  = nn.Sequential(nn.Linear(dd, 128), nn.ReLU(), nn.Linear(128, 1))
    def forward(self, obs, theta, h0=None):
        out, hT = self.gru(obs, h0)
        feat = torch.cat([out, self.theta_enc(theta)], dim=-1)
        return self.pi(feat), self.v(feat).squeeze(-1), hT

## Loaders and defender wrappers (shared by best_response, eval, and outcomes)

In [ ]:
def Load(net, path):
    net.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return net.eval()


def Load_Baselines(save_dir="compared"):
    # The five trained baselines from baselines.ipynb
    ck_ = lambda name: torch.load(os.path.join(save_dir, name + ".pth"), map_location=device, weights_only=True)
    out = {}
    for name in ('RPPO', 'DR_RPPO'):
        ck = ck_(name)
        pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
        out[name] = {'policy': pol}
    ck = ck_("VARIBAD"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(2 * L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = VariBADEncoder(L, ck['config']['RNN_H']).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['VARIBAD'] = {'policy': pol, 'encoder': enc, 'latent_dim': L}
    ck = ck_("PEARL"); L = ck['config']['LATENT_DIM']
    pol = RecurrentAC(L, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['policy']); pol.eval()
    enc = PEARLEncoder(L).to(device); enc.load_state_dict(ck['encoder']); enc.eval()
    out['PEARL'] = {'policy': pol, 'encoder': enc, 'latent_dim': L, 'resample_k': ck['config'].get('resample_k', 10)}
    ck = ck_("MMD")
    pol = RecurrentAC(0, ck['config']['RNN_H']).to(device); pol.load_state_dict(ck['p1']); pol.eval()
    out['MMD'] = {'policy': pol}
    return out
#################################################################################################################
# Defender wrappers: reset(...) at the start of an episode, act_dist(x, p_row, p_col) for the move distribution,
# observe(a, u, erc, new_prc, new_erc, y) after each transition. They use the globals Context_Model, critic,
# Player1, and BASELINES, which each evaluation notebook loads.
class PipelineDefender:
    # 'Trained P1': our pursuer on [z^S, z^Theta]. 'Critic P1': the Oracle, the critic's equilibrium row
    # with privileged access to the state and context.
    def __init__(self, name):
        self.name = name
    def reset(self, Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col):
        self.th = torch.tensor(Theta, dtype=torch.float32, device=device)
        self.tt = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        self.z_s = init_z_s(p_row, p_col)
        self.Context = np.zeros([max_steps, CTX_DIM], dtype=np.float32)
        self.m2 = init_m2(e_row, e_col)
        self.t = 0
    def act_dist(self, x, p_row, p_col):
        ctx = torch.tensor(self.Context, dtype=torch.float32, device=device).view(1, max_steps, CTX_DIM)
        z_s_t = torch.tensor(self.z_s, dtype=torch.float32, device=device)
        with torch.no_grad():
            Emb = Context_Model(ctx, Context_Mask(self.t)).squeeze(0)
            if self.name == 'Trained P1':
                p = Player1(z_s_t, Emb).cpu().numpy()
            else:
                Q = critic(z_s_t, Emb, torch.tensor(x, dtype=torch.float32, device=device),
                           torch.tensor(self.m2, dtype=torch.float32, device=device),
                           self.tt, self.th).cpu().numpy().reshape(n_a, n_u)
                _, p, _ = MatrixValue(Q)
        p = np.clip(np.asarray(p, dtype=np.float64), 0, None)
        return p / p.sum()
    def observe(self, a, u, erc, new_prc, new_erc, y):
        if self.name == 'Critic P1':
            self.m2 = m2_update(self.m2, erc[0], erc[1], u, new_erc[0], new_erc[1])
        next_z_s = Def_State_Blf(self.z_s, y, new_prc[0], new_prc[1])
        self.Context[self.t, :] = Context_Row(self.z_s, a, next_z_s)
        self.z_s = next_z_s
        self.t += 1


class RandomDefender:
    def reset(self, Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col):
        pass
    def act_dist(self, x, p_row, p_col):
        return np.full(n_a, 1.0 / n_a)
    def observe(self, a, u, erc, new_prc, new_erc, y):
        pass


class BaselineDefender:
    # A trained baseline acting on [own position, y, a_prev], with VariBAD's belief or PEARL's sampled latent as extra input
    def __init__(self, name):
        self.name = name
        self.m = BASELINES[name]
    def reset(self, Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col):
        self.h = None
        self.y_cur = np.zeros(d)
        self.a_prev = np.zeros(n_a)
        if self.name == 'VARIBAD':
            self.enc_h = None
        if self.name == 'PEARL':
            self.ctx = []
            self.z = np.random.randn(self.m['latent_dim']).astype(np.float32)
    def _extra(self, obs):
        if self.name == 'VARIBAD':
            with torch.no_grad():
                mu, logvar, self.enc_h = self.m['encoder'](
                    torch.tensor(obs, dtype=torch.float32, device=device).view(1, 1, -1), self.enc_h)
            mu = mu.view(-1).cpu().numpy()
            std = np.exp(0.5 * logvar.view(-1).cpu().numpy())
            return np.concatenate([mu, std])
        if self.name == 'PEARL':
            return self.z
        return None
    def act_dist(self, x, p_row, p_col):
        own = np.zeros(d); own[p_row * 6 + p_col] = 1.0
        obs = np.concatenate([own, self.y_cur, self.a_prev]).astype(np.float32)
        ex_np = self._extra(obs)
        obs_t = torch.tensor(obs, device=device).view(1, 1, OBS_DIM)
        ex_t = None if ex_np is None else torch.tensor(np.asarray(ex_np, dtype=np.float32), device=device).view(1, 1, -1)
        with torch.no_grad():
            logits, _, self.h = self.m['policy'](obs_t, ex_t, self.h)
        p = F.softmax(logits.view(-1), dim=-1).cpu().numpy().astype(np.float64)
        return p / p.sum()
    def observe(self, a, u, erc, new_prc, new_erc, y):
        a_onehot = np.zeros(n_a, dtype=np.float32); a_onehot[a] = 1.0
        if self.name == 'PEARL':
            self.ctx.append(np.concatenate([self.y_cur, a_onehot, y]).astype(np.float32))
            if len(self.ctx) % self.m['resample_k'] == 0:
                with torch.no_grad():
                    mu, var = self.m['encoder'].posterior(torch.tensor(np.stack(self.ctx), device=device))
                self.z = (mu + var.sqrt() * torch.randn_like(mu)).cpu().numpy()
        self.y_cur = y
        self.a_prev = a_onehot


def get_defender(name):
    if name in ('Critic P1', 'Trained P1'):
        return PipelineDefender(name)
    if name == 'Random P1':
        return RandomDefender()
    return BaselineDefender(name)